# Session 3.1 -- Naive RAG

**AI-2: AI Backend Engineering**

---

## Learning Objectives

By the end of this session, you will be able to:

1. **Explain** the Retrieve-Augment-Generate pattern and articulate why retrieval quality bounds answer quality
2. **Build** the augmentation and orchestration stages of a naive RAG pipeline -- `build_prompt()` and `naive_rag()` -- by wiring together retrieval, prompt construction, and generation
3. **Identify** semantic similarity pitfalls -- semantic drift, keyword-heavy queries, ambiguous intent, and the "lost in the middle" problem -- and predict when naive RAG will fail

## What You Are Working With

- `src/s4_retrieval/naive.py` -- Naive RAG module with `naive_retrieve()` and `RAGResponse` (provided complete)
- `src/s0_generation/generate.py` -- Claude API functions from Session 1.1 (provided complete)
- `src/s2_embeddings/embed.py` -- Voyage AI embedding functions from Session 2.1 (provided complete)
- `src/s3_ingestion/store.py` -- ChromaDB storage from Session 2.2 (provided complete)
- `data/northbrook/` -- Corpus of memos, meeting notes, financial reports, and policies from Northbrook Partners

You **import and use** the pre-built retrieval function. You **build** the prompt construction and pipeline orchestration yourself.

**Prerequisite:** Your ChromaDB collection must be populated from Session 2.2. If it is empty, run the ingestion notebook first.

---

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Find repo root by walking up until we find pyproject.toml
_here = Path(".").resolve()
_root = _here
while _root != _root.parent:
    if (_root / "pyproject.toml").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))

from dotenv import load_dotenv
load_dotenv()

In [ ]:
from src.s4_retrieval.naive import naive_retrieve, RAGResponse
from src.s0_generation.generate import call_claude_with_usage
from src.s3_ingestion.store import get_collection

print("All imports loaded successfully.")

In [ ]:
# Verify ChromaDB is populated from Session 2.2
collection = get_collection()
count = collection.count()
print(f"Collection '{collection.name}' contains {count} chunks.")

if count == 0:
    print("\nWARNING: Your collection is empty!")
    print("Run the Session 2.2 notebook first to populate ChromaDB.")
else:
    print(f"\nReady to go. {count} chunks available for retrieval.")

---

## 2. RAG Overview -- Retrieve, Augment, Generate

RAG stands for **Retrieval Augmented Generation**. The idea is simple: before asking a model to answer a question, **look up relevant information first** and include it in the prompt.

- **Not fine-tuning** -- we do not change the model
- **Not prompt engineering alone** -- we dynamically select what goes into the prompt
- RAG = "look it up, then answer" -- the model becomes a reasoning engine over YOUR data

### The Pipeline

```
                  +-----------+
  Question -----> | RETRIEVE  |  Embed the question, query ChromaDB
                  +-----------+  for top-k most similar chunks
                       |
                       v
                  +-----------+
    Chunks -----> | AUGMENT   |  Inject retrieved chunks into a
                  +-----------+  structured prompt with grounding
                       |         instructions
                       v
                  +-----------+
   Prompt  -----> | GENERATE  |  Send the augmented prompt to Claude
                  +-----------+  and return the answer + sources
                       |
                       v
               Answer + Sources
```

### Three Truths About RAG

These came from our Session 2.2 closing questions:

1. **Similar != Useful** -- Semantic similarity measures direction in embedding space, not usefulness for a specific question
2. **Context present != Answer grounded** -- Claude will try to answer even if the context is insufficient, filling gaps from training data
3. **Trust requires evidence** -- You must see the retrieved chunks, their scores, and compare the answer against the source text

### Why "Naive"?

Naive RAG = pure semantic similarity retrieval, no filtering, no re-ranking, no query transformation. Every query gets the same retrieval logic regardless of question type. This is the baseline. Session 3.2 raises the ceiling with metadata filtering.

---

## 3. First RAG Query

Let's run the full pipeline end-to-end. One function call: question in, grounded answer out.

In [ ]:
# Demo: see the full pipeline in action
# (The module has a complete version -- you'll build your own in Section 4)
from src.s4_retrieval.naive import naive_rag as _demo_rag

response = _demo_rag("What is Northbrook Partners' vacation policy?")

print("=" * 70)
print("QUESTION")
print("=" * 70)
print(response.question)
print()
print("=" * 70)
print("ANSWER")
print("=" * 70)
print(response.answer)

In [ ]:
# Inspect the RAGResponse object -- everything is visible
print(f"Model:         {response.model}")
print(f"Input tokens:  {response.input_tokens}")
print(f"Output tokens: {response.output_tokens}")
print(f"Sources:       {len(response.sources)} chunks retrieved")
print()

# List the sources with their similarity scores
print("--- Retrieved Sources ---")
for i, src in enumerate(response.sources):
    source_name = src["metadata"].get("source", "unknown")
    doc_type = src["metadata"].get("doc_type", "unknown")
    score = src["score"]
    print(f"  [{i+1}] {score:.4f}  {source_name}  (type: {doc_type})")

print()
print("Higher scores mean the chunk is more similar to the question.")
print("But similar does not guarantee useful -- we will test that.")

In [ ]:
# Look at the actual text of the top source
top_source = response.sources[0]

print(f"--- Top Source (score: {top_source['score']:.4f}) ---")
print(f"From: {top_source['metadata'].get('source', 'unknown')}")
print()
print(top_source["text"])

**What just happened:**

1. The question was embedded using Voyage AI (same model that embedded the documents)
2. ChromaDB returned the 5 most similar chunks
3. Those chunks were formatted into a prompt with grounding instructions
4. Claude read the chunks and produced an answer citing the sources
5. The response includes the answer, sources, scores, and token counts

That is RAG. Now let's look under the hood.

---

## 4. Under the Hood -- Build the Pipeline

You've seen `naive_rag()` work end-to-end. Now let's call each stage separately. The retrieve stage (`naive_retrieve()`) is provided -- it handles ChromaDB plumbing. **You will build the augmentation and orchestration stages yourself.**

The map: call each stage to see what it does. The territory: implement the two functions that tie the stages together.

### Stage 1: Retrieve

`naive_retrieve()` embeds the question and queries ChromaDB for the top-k most similar chunks.

In [ ]:
# Step 1: Retrieve chunks
question = "What is Northbrook Partners' vacation policy?"
sources = naive_retrieve(question, top_k=5)

print(f"Retrieved {len(sources)} chunks for: '{question}'\n")

for i, src in enumerate(sources):
    score = src["score"]
    source_name = src["metadata"].get("source", "unknown")
    doc_type = src["metadata"].get("doc_type", "unknown")
    chunk_idx = src["metadata"].get("chunk_index", "?")
    text_preview = src["text"][:150].replace("\n", " ")

    print(f"--- Chunk {i+1} [similarity: {score:.4f}] ---")
    print(f"Source: {source_name}  |  Type: {doc_type}  |  Chunk: {chunk_idx}")
    print(f"Text:   {text_preview}...")
    print()

**What to notice:**

- The chunks are sorted by similarity score (highest first)
- ChromaDB returns cosine distances. The module converts them: **similarity = 1 - distance**
- Some chunks are highly relevant. Others may be topically related but not directly useful
- The metadata tells you exactly which document each chunk came from

**Key question:** Are ALL of these chunks actually useful for answering the question? Or did some drift in because they share vocabulary?

### Stage 2: Augment -- Your First Build Task

You've seen the retrieved chunks. Now build the function that turns them into a grounded prompt for Claude.

`build_prompt()` is the bridge between retrieval and generation. It defines what Claude sees and how Claude behaves.

### Your Turn: Build the RAG Prompt

You've seen the retrieved chunks. Now build the prompt that Claude will receive.

Your `build_prompt()` function should:
1. Define a **system prompt** telling Claude to answer using ONLY the provided context, cite sources, and say "I don't have enough information..." when context is insufficient
2. **Format each source** as a labeled block: `[Source: {name}, Score: {score}]` followed by the text
3. **Join** all source blocks with `\n\n---\n\n` separators
4. **Build the user message**: `Context:\n\n{context}\n\n---\n\nQuestion: {question}`
5. Return `(system_prompt, user_message)`

The functions you need are already imported. The `sources` list contains dicts with keys: `"text"`, `"metadata"` (dict with `"source"` key), and `"score"`.

In [ ]:
def build_prompt(question: str, sources: list[dict]) -> tuple[str, str]:
    """Build the system prompt and user message for RAG.

    Args:
        question: The user's question.
        sources: Retrieved chunks from naive_retrieve().
                 Each dict has: "text", "metadata" (with "source" key), "score"

    Returns:
        A tuple of (system_prompt, user_message).
    """
    # TODO: Implement this function
    # Hint: source name is at source["metadata"].get("source", "Unknown")
    # Hint: format scores with f"{score:.3f}"
    pass

In [ ]:
# Verify: if your build_prompt() works, you should see formatted sources below
system_prompt, user_message = build_prompt(question, sources)

print("=" * 70)
print("SYSTEM PROMPT")
print("=" * 70)
print(system_prompt)
print()
print("=" * 70)
print("USER MESSAGE (first 1000 characters)")
print("=" * 70)
print(user_message[:1000])
if len(user_message) > 1000:
    print(f"\n... [{len(user_message) - 1000} more characters]")

**Nice work.** You just built the augmentation stage of RAG. Read through the prompt carefully -- this is what Claude actually sees. Notice:

- The **system prompt** tells Claude to answer using ONLY the provided context and to say "I don't have enough information" when context is insufficient. This is the grounding instruction.
- Each chunk includes its **source name** and **similarity score** in the header -- this lets Claude cite sources.
- The chunks are separated by `---` dividers for clarity.
- The question appears **after** the context. Claude reads the evidence before answering.

Without the grounding instruction, Claude will happily fill in gaps from its training data. With it, Claude is more likely to say "I don't know" when the context is insufficient.

### Stage 3: Generate -- Wire the Full Pipeline

You have the retrieve stage (`naive_retrieve()`) and you just built the augment stage (`build_prompt()`). Now wire everything together into `naive_rag()`.

**Generation is the easy part. The hard part is everything before it.**

### Your Turn: Wire the Full RAG Pipeline

You have all three stages working:
- **Retrieve**: `naive_retrieve(question, top_k)` → list of source dicts
- **Augment**: your `build_prompt(question, sources)` → (system_prompt, user_message)
- **Generate**: `call_claude_with_usage(prompt=..., system_prompt=..., temperature=0.0)` → returns a dict:
  - `"text"` — Claude's response text
  - `"input_tokens"` — tokens used by the prompt
  - `"output_tokens"` — tokens used by the response
  - `"model"` — the model that handled the request

Wire them together into `naive_rag()`. Handle the edge case where retrieval returns nothing.

In [ ]:
def naive_rag(question: str, top_k: int = 5) -> RAGResponse:
    """Run the full naive RAG pipeline: retrieve → augment → generate.

    Args:
        question: The user's question.
        top_k: Number of chunks to retrieve (default: 5).

    Returns:
        A RAGResponse with the answer, sources, and token usage.
    """
    # TODO: Implement this function
    # 1. Retrieve sources with naive_retrieve()
    # 2. If no sources: return RAGResponse with "No relevant documents found."
    #    and zeros for token counts, empty string for model
    # 3. Build prompt with your build_prompt()
    # 4. Call Claude with call_claude_with_usage()
    # 5. Return a RAGResponse packing: question, answer, sources, tokens, model
    pass

In [ ]:
# Test your pipeline
test_response = naive_rag("What is Northbrook Partners' vacation policy?")
print(f"Question: {test_response.question}")
print(f"Answer:   {test_response.answer[:200]}...")
print(f"Sources:  {len(test_response.sources)}")
print(f"Tokens:   {test_response.input_tokens} in / {test_response.output_tokens} out")

---

## 5. Answer vs Evidence Panel

How do you verify that an AI-generated answer actually came from your sources? You compare them side by side.

The helper function below takes a `RAGResponse` and prints Claude's answer alongside the actual chunk text. This makes unsupported claims immediately visible.

In [ ]:
import textwrap


def compare_answer_to_evidence(rag_response: RAGResponse) -> None:
    """Display Claude's answer alongside the actual retrieved evidence.

    This helper makes it easy to spot unsupported claims. If a claim
    appears in the answer but NOT in any chunk, Claude made it up.

    Args:
        rag_response: A RAGResponse from naive_rag() or similar.
    """
    divider = "=" * 70

    print(divider)
    print("QUESTION")
    print(divider)
    print(rag_response.question)
    print()

    print(divider)
    print("CLAUDE'S ANSWER")
    print(divider)
    # Break the answer into individual sentences for easier comparison
    answer_text = rag_response.answer
    print(answer_text)
    print()

    print(divider)
    print("RETRIEVED EVIDENCE (what Claude was given)")
    print(divider)
    for i, src in enumerate(rag_response.sources):
        source_name = src["metadata"].get("source", "unknown")
        score = src["score"]
        print(f"\n--- Source {i+1}: {source_name} [score: {score:.4f}] ---")
        # Wrap text for readability
        wrapped = textwrap.fill(src["text"], width=80)
        print(wrapped)

    print()
    print(divider)
    print("VERIFICATION CHECKLIST")
    print(divider)
    print("For each claim in the answer above, check:")
    print("  [?] Does this claim appear in one of the source chunks?")
    print("  [?] Is the claim accurately represented (not taken out of context)?")
    print("  [?] Are there claims in the answer with NO supporting evidence above?")
    print()
    print("If a claim appears in the answer but NOT in any chunk -- Claude made it up.")
    print("If a claim merges information from chunks that should not be combined --")
    print("that is also a hallucination.")

In [ ]:
# Run it on our first response -- the vacation policy question
compare_answer_to_evidence(response)

**Walk through the comparison above.**

- Read each claim in Claude's answer
- Find the supporting text in the retrieved sources
- Are there any claims that do NOT appear in any source?

For a straightforward factual question like the vacation policy, the answer should be well-grounded. But keep this helper ready -- we are about to see cases where the comparison reveals serious problems.

---

## 6. The Confidence Mirage

This is the most important demo of the session.

Low similarity scores are easy to catch -- you see the problem immediately. The dangerous failures are the ones where:

- Similarity scores are **high** (0.80+)
- Claude produces a **confident, well-cited** answer
- The answer is **wrong**

### The Setup

The Northbrook corpus contains three documents relevant to PTO:

1. **vacation_policy_2025.md** -- The new 2025 policy that says the carryover limit is **10 days** (increased from 4)
2. **memo_policy_correction.md** -- A correction memo from February 2025 stating that the **old 4-day limit still applies for 2025**, and the 10-day cap does not take effect until January 1, 2026
3. **vacation_policy_2023.md** -- The old policy with a **4-day carryover limit** and 17 vacation days

What happens when we ask about the current carryover limit?

In [ ]:
# THE CONFIDENCE MIRAGE
#
# This query is designed to expose the most dangerous RAG failure:
# high scores + confident answer + wrong

mirage_query = "What is the PTO carryover limit for 2025?"
mirage_response = naive_rag(mirage_query)

print("=" * 70)
print("THE CONFIDENCE MIRAGE")
print("=" * 70)
print(f"\nQuestion: {mirage_query}")
print(f"\nAnswer:")
print(mirage_response.answer)
print()
print("--- Sources ---")
for i, src in enumerate(mirage_response.sources):
    print(f"  [{i+1}] {src['score']:.4f}  {src['metadata'].get('source', 'unknown')}")

**Read the answer above carefully.**

- Are the similarity scores high? (They should be -- all these documents are about PTO.)
- Does Claude sound confident?
- Does Claude cite sources?

Now let's look at what the chunks actually say.

In [ ]:
# Now compare the answer to the actual evidence
compare_answer_to_evidence(mirage_response)

### What Went Wrong?

The corpus contains **conflicting information** about the carryover limit:

| Document | Says |
|----------|------|
| `vacation_policy_2025.md` | Carryover limit is **10 days** (increased from 4) |
| `memo_policy_correction.md` | The 10-day cap does **NOT** take effect until Jan 1, 2026. For 2025, the old **4-day limit** still applies |
| `vacation_policy_2023.md` | Carryover limit is **4 days** |

**The correct answer for 2025 is 4 days** (per the correction memo). But naive RAG retrieves chunks from all three documents because they are all semantically similar to the question. Claude then has to reconcile conflicting context.

The typical failure mode: Claude either picks the 10-day number (from the 2025 policy) because it appears to be the newer document, or merges information from multiple sources without flagging the contradiction.

**This is the Confidence Mirage:**
- High similarity scores (the retrieval "worked")
- Confident, well-cited answer (Claude "did its job")
- **Wrong answer** (the system failed silently)

The scores did not warn you. The citations did not warn you. The only way to catch this is to **read the chunks and compare them to the answer**.

In [ ]:
# Let's look at the retrieval more closely
# Which documents did naive_retrieve pull back, and what do they say?

mirage_sources = naive_retrieve(mirage_query, top_k=5)

print(f"Retrieval results for: '{mirage_query}'\n")

for i, src in enumerate(mirage_sources):
    source_name = src["metadata"].get("source", "unknown")
    score = src["score"]
    print(f"=== Chunk {i+1}: {source_name} [score: {score:.4f}] ===")
    print(src["text"])
    print()

**Key takeaway:** Naive RAG treats all documents equally. It has no way to know that the correction memo overrides the policy document. It has no way to reason about recency, authority, or document relationships. It just finds the most similar chunks and hands them to Claude.

In production, this is how you get confident, well-sourced, **wrong** answers. The dangerous failures are the ones that look correct.

---

## 7. Failure Gallery

Now it is your turn. Below are five failure categories. Your goal: **find at least 3 failures from at least 3 different categories.**

For each failure, document:
1. The **question** you asked
2. The **top-3 retrieved chunks** (source and score)
3. **Claude's answer**
4. Your **analysis** of what went wrong

Use `compare_answer_to_evidence()` to inspect each one.

**These failures become your Lab 2 test set.** Take them seriously.

### Failure Categories

| Category | Description | What to Try |
|----------|-------------|-------------|
| **Semantic Drift** | Chunks are topically related but do not answer the question | Questions about specific details when corpus has general overviews; different vocabulary ("PTO" vs "vacation") |
| **Keyword-Heavy Queries** | Terms in the question skew results toward wrong chunks | Questions mentioning proper nouns that appear in irrelevant documents; ambiguous terms |
| **Hallucination Despite Context** | Claude produces an answer not supported by the chunks | Questions where chunks have partial info; chunks from different time periods that Claude merges |
| **Top-K Sensitivity** | Answer quality changes significantly with different k values | Same question with top_k=1 vs 3 vs 10; questions where more chunks introduce noise |
| **Unanswerable Questions** | The corpus genuinely cannot answer this | Questions about topics not in the corpus; specific dates/people/numbers not in any document |

### Failure 1: Wrong Version

When multiple versions of a document exist, naive RAG may pull the wrong one -- or merge them.

In [ ]:
# Failure 1: Wrong version
# The corpus has vacation_policy_2023.md AND vacation_policy_2025.md
# Which one does naive RAG retrieve?

wrong_version_query = "How many vacation days do employees get?"
wrong_version_response = naive_rag(wrong_version_query)

print(f"Question: {wrong_version_query}\n")
print(f"Answer:\n{wrong_version_response.answer}\n")
print("--- Sources ---")
for i, src in enumerate(wrong_version_response.sources):
    print(f"  [{i+1}] {src['score']:.4f}  {src['metadata'].get('source', 'unknown')}")

In [ ]:
# Side-by-side comparison
compare_answer_to_evidence(wrong_version_response)

**Your analysis:** (Write your observations here)

- Which policy versions appeared in the results?
- Did Claude merge information from the 2023 and 2025 policies?
- What is the correct answer? (2025 policy: 20 days; 2023 policy: 17 days, or 20 with 5+ years tenure)
- What would you need to fix this? (Hint: what if you could filter by year?)

### Failure 2: Temporal Confusion

Three documents contain conflicting information about the same topic from different time periods.

In [ ]:
# Failure 2: Temporal confusion -- 3-way conflict
# vacation_policy_2023.md: 4-day carryover, 17 days vacation
# vacation_policy_2025.md: 10-day carryover, 20 days vacation
# memo_policy_correction.md: actually still 4-day carryover for 2025

temporal_query = "What is the maximum number of vacation days I can carry over to next year?"
temporal_response = naive_rag(temporal_query)

print(f"Question: {temporal_query}\n")
print(f"Answer:\n{temporal_response.answer}\n")
print("--- Sources ---")
for i, src in enumerate(temporal_response.sources):
    print(f"  [{i+1}] {src['score']:.4f}  {src['metadata'].get('source', 'unknown')}")

In [ ]:
compare_answer_to_evidence(temporal_response)

**Your analysis:** (Write your observations here)

- Did Claude acknowledge the contradiction between documents?
- What number did Claude give? Was it correct?
- The **correct** answer depends on timing: for 2025 the carryover limit is still 4 days; starting 2026, it becomes 10 days
- Naive RAG has no concept of "which document is authoritative" or "which document is more recent"

### Failure 3: Missing Context (Answer Spans Multiple Chunks)

What happens when the answer requires information from two chunks that are not both in the top-k?

In [ ]:
# Failure 3: Missing context
# The professional development budget ($3,200) is in the employee handbook.
# But the expense reimbursement process is in the expense policy.
# A question that spans both may only retrieve one.

missing_context_query = "How do I get reimbursed for a professional development conference?"
missing_context_response = naive_rag(missing_context_query)

print(f"Question: {missing_context_query}\n")
print(f"Answer:\n{missing_context_response.answer}\n")
print("--- Sources ---")
for i, src in enumerate(missing_context_response.sources):
    print(f"  [{i+1}] {src['score']:.4f}  {src['metadata'].get('source', 'unknown')}")

In [ ]:
compare_answer_to_evidence(missing_context_response)

**Your analysis:** (Write your observations here)

- Did the retrieval pull chunks from both the employee handbook AND the expense policy?
- Does the answer combine the budget amount ($3,200) with the reimbursement process?
- If one document was missing from the results, did Claude fill in the gap or acknowledge it?
- This is the "information scattered across documents" problem -- naive RAG retrieves by similarity, not by completeness

### Failure 4: Over-Retrieval (Vague Query)

What happens when the query is too broad and retrieves chunks from many unrelated documents?

In [ ]:
# Failure 4: Over-retrieval with a vague query
# A broad question that could match many documents

vague_query = "What are the important company policies?"
vague_response = naive_rag(vague_query)

print(f"Question: {vague_query}\n")
print(f"Answer:\n{vague_response.answer}\n")
print("--- Sources ---")
for i, src in enumerate(vague_response.sources):
    source_name = src["metadata"].get("source", "unknown")
    doc_type = src["metadata"].get("doc_type", "unknown")
    print(f"  [{i+1}] {src['score']:.4f}  {source_name}  (type: {doc_type})")

In [ ]:
compare_answer_to_evidence(vague_response)

**Your analysis:** (Write your observations here)

- How many different documents appeared in the sources?
- Are the similarity scores all relatively close (indicating no chunk is clearly "the right one")?
- Is the answer a shallow summary that does not deeply address any one topic?
- What would make this question answerable? (More specific query, narrower scope)

### Failure 5: Unanswerable Question

The corpus does not contain the information needed to answer this question. Does Claude admit it?

In [ ]:
# Failure 5: Unanswerable question -- topic not in the corpus
unanswerable_query = "What was the company's Q3 2024 revenue?"
unanswerable_response = naive_rag(unanswerable_query)

print(f"Question: {unanswerable_query}\n")
print(f"Answer:\n{unanswerable_response.answer}\n")
print("--- Sources ---")
for i, src in enumerate(unanswerable_response.sources):
    print(f"  [{i+1}] {src['score']:.4f}  {src['metadata'].get('source', 'unknown')}")

print("\nNotice the similarity scores. Are they lower than for answerable questions?")
print("Did Claude say 'I don't have enough information' or did it fabricate an answer?")

In [ ]:
compare_answer_to_evidence(unanswerable_response)

**Your analysis:** (Write your observations here)

- Did Claude say "I don't have enough information" or did it fabricate a number?
- What were the similarity scores? Were they noticeably lower?
- Did the grounding instruction in the system prompt do its job?
- If Claude DID give a number, where did it come from? (Its training data, not your corpus.)

---

## 8. Retrieval Anatomy

The Failure Gallery showed you *what* breaks. Now let's look at *why*.

In Session 2.1, you computed cosine similarity between pairs of sentences — a single number. Now you'll see those scores **in action as a retrieval gate**: everything above a threshold gets into Claude's context window, everything below doesn't.

Two questions drive this section:

1. **How spread out are the scores?** If scores are tightly clustered, the system can't clearly distinguish relevant from irrelevant chunks.
2. **Which documents dominate retrieval?** If one doc has more chunks, it gets more "votes" — regardless of whether it's the right doc.

In [ ]:
# Score Landscape: What does retrieval actually look like?
# Let's pull back 15 chunks for a common query and see the full picture.

landscape_query = "How many vacation days do employees get?"
sources_15 = naive_retrieve(landscape_query, top_k=15)

scores = [s["score"] for s in sources_15]

print(f"Query: {landscape_query}\n")
print(f"  {'Rank':<6} {'Score':<9} {'':50} {'Source'}")
print(f"  {'-'*100}")
for i, src in enumerate(sources_15):
    score = src["score"]
    source_name = src["metadata"].get("source", "unknown")
    bar = "█" * int(score * 80)
    print(f"  {i+1:>2}.   {score:.4f}   {bar:<50} {source_name}")

gap_1_5 = scores[0] - scores[4]
gap_1_10 = scores[0] - scores[9]
print(f"\n  Score gap (rank 1 → rank 5):   {gap_1_5:.4f}")
print(f"  Score gap (rank 1 → rank 10):  {gap_1_10:.4f}")
print(f"\n  At top_k=5, Claude sees ranks 1-5. At top_k=10, ranks 1-10.")
print(f"  How confident are you that rank 5 is more relevant than rank 6?")

In [ ]:
# Now contrast with a SPECIFIC named query
# "Curiosity Credit" is a named program in the employee handbook

specific_query = "What is the Curiosity Credit?"
specific_sources = naive_retrieve(specific_query, top_k=15)

print(f"Query: {specific_query}\n")
print(f"  {'Rank':<6} {'Score':<9} {'':50} {'Source'}")
print(f"  {'-'*100}")
for i, src in enumerate(specific_sources):
    score = src["score"]
    source_name = src["metadata"].get("source", "unknown")
    bar = "█" * int(score * 80)
    print(f"  {i+1:>2}.   {score:.4f}   {bar:<50} {source_name}")

specific_scores = [s["score"] for s in specific_sources]
print(f"\n  Score gap (rank 1 → rank 5):  {specific_scores[0] - specific_scores[4]:.4f}")

# Source concentration: which documents dominate retrieval?
print(f"\n{'='*70}")
print("SOURCE CONCENTRATION — Who gets the most votes?")
print(f"{'='*70}")
print(f"\nQuery: {landscape_query}")
print(f"\n  {'Source Document':<40} {'Chunks':>7} {'Avg Score':>10} {'Max Score':>10}")
print(f"  {'-'*70}")

by_source = {}
for src in sources_15:
    name = src["metadata"].get("source", "?")
    by_source.setdefault(name, []).append(src["score"])

for name, doc_scores in sorted(by_source.items(), key=lambda x: max(x[1]), reverse=True):
    avg = sum(doc_scores) / len(doc_scores)
    print(f"  {name:<40} {len(doc_scores):>7} {avg:>10.4f} {max(doc_scores):>10.4f}")

print(f"\n  The 2023 policy has MORE chunks than the 2025 policy.")
print(f"  More chunks = more chances to appear in top-k. Volume wins over recency.")

**What you just saw:**

**Vague query ("How many vacation days..."):**
- Scores are tightly clustered. The gap between rank 1 and rank 5 is small — the system can't clearly tell relevant from irrelevant.
- The **2023 policy dominates** retrieval because it has more chunks. More chunks = more votes in semantic search. The outdated document wins by volume, not by relevance.
- This is your chunking strategy from Session 2.2 directly affecting retrieval quality.

**Specific query ("What is the Curiosity Credit?"):**
- Rank 1 has a **clear lead** — the gap between rank 1 and rank 2 is large. The system knows exactly where to look.
- Named concepts with distinctive vocabulary produce strong, unambiguous signals.

**The pattern:** Embedding models handle specific, named concepts well. They struggle with broad questions where many documents share vocabulary. This is exactly the embedding fails lesson from Session 2.1 — now you're seeing it break a real pipeline.

In [ ]:
# Top-K Deep Dive: k is a correctness knob, not just a cost knob.
#
# Three queries, four k values. Watch which documents make it in — and which don't.

topk_test = [
    {"question": "What is the company's AI strategy?",
     "expected": "memo_ceo_annual_kickoff.md", "label": "Strong signal"},
    {"question": "How many vacation days do employees get?",
     "expected": "vacation_policy_2025.md", "label": "Temporal trap"},
    {"question": "How many days of vacation carryover are allowed?",
     "expected": "vacation_policy_2025.md", "label": "Barely in range"},
]

k_values = [1, 3, 5, 10]

print("TOP-K DEEP DIVE")
print("=" * 95)
print(f"  {'Query':<50} {'k':>3} {'Hit?':>5} {'Rank':>5} {'Docs':>5} {'Tokens':>7} {'Min Score':>10}")
print(f"  {'-'*88}")

for tq in topk_test:
    for k in k_values:
        sources = naive_retrieve(tq["question"], top_k=k)
        rank = None
        for i, src in enumerate(sources):
            if src["metadata"].get("source") == tq["expected"]:
                rank = i + 1
                break
        unique_docs = len(set(s["metadata"].get("source", "?") for s in sources))
        total_tokens = sum(s["metadata"].get("token_count", 0) for s in sources)
        min_score = min(s["score"] for s in sources) if sources else 0
        hit = "Yes" if rank else "MISS"
        rank_str = str(rank) if rank else "--"
        q_short = tq["question"][:48]
        print(f"  {q_short:<50} {k:>3} {hit:>5} {rank_str:>5} {unique_docs:>5} {total_tokens:>7} {min_score:>10.4f}")
    print()

**What the Top-K table reveals:**

- **AI strategy** (strong signal): Hits at rank 1 regardless of k. When the embedding model has a clear match, more chunks just add noise without changing the answer. Token cost goes from ~85 to ~1,200.
- **Vacation days** (temporal trap): **MISS at k=1** — at k=1, you get the WRONG year (2023 beats 2025). You need k=3+ just to get the right document into the context. But even then, it's rank 2, not rank 1.
- **Vacation carryover** (barely in range): **MISS at k=1 AND k=3**. The 2025 policy doesn't appear until rank 5. If you'd set top_k=3, your pipeline would never see the correct document.

**k is not just a cost knob. It's a correctness knob.** Too small and you miss documents. Too large and you pay for noise. There is no single right answer — it depends on the query.

Look at the Tokens column: going from k=1 to k=10 costs roughly 10x more tokens per query. If you're running 1,000 queries a day, the difference between k=3 and k=10 is real money.

---

## 9. Scoring Retrieval Quality

So far we have been eyeballing results. Can we put a number on retrieval quality?

Two simple metrics:
- **Hit Rate**: Did the right document appear anywhere in the top-k? (If it's not there, the answer CAN'T be correct.)
- **Mean Reciprocal Rank (MRR)**: How high did it rank? (Rank 1 = perfect. Rank 5 = technically there but buried.)

Below is a test set that mixes easy questions, vocabulary mismatches, and ambiguous queries. Not everything will hit.

In [ ]:
# Mixed test set: easy hits + vocabulary mismatches + ambiguous queries.
# Some of these WILL miss. That's the point.

test_queries = [
    # Easy — direct vocabulary match
    {"question": "What is the dress code?", "expected_source": "employee_handbook.md"},
    {"question": "What is the meal reimbursement limit?", "expected_source": "expense_policy.md"},
    {"question": "What is the company's AI strategy?", "expected_source": "memo_ceo_annual_kickoff.md"},
    {"question": "What is the remote work policy?", "expected_source": "remote_work_policy.md"},
    # Vocabulary mismatch — asking with different words than the source uses
    {"question": "What are the WFH rules?",  "expected_source": "remote_work_policy.md"},
    {"question": "How much PTO do new hires get?", "expected_source": "vacation_policy_2025.md"},
    # Ambiguous / barely in range
    {"question": "What changed in the latest vacation policy?", "expected_source": "memo_policy_correction.md"},
    {"question": "What initiatives are planned for Q1 2025?", "expected_source": "engineering_standup_2025_01.md"},
]

print(f"Running {len(test_queries)} test queries (top_k=5)...\n")
print(f"  {'Question':<50} {'Expected Source':<35} {'Rank':>5} {'Hit?':>5}")
print(f"  {'-'*98}")

hits = 0
reciprocal_ranks = []

for test in test_queries:
    sources = naive_retrieve(test["question"], top_k=5)
    rank = None
    for i, src in enumerate(sources):
        if src["metadata"].get("source") == test["expected_source"]:
            rank = i + 1
            break

    hit = rank is not None
    if hit:
        hits += 1
        reciprocal_ranks.append(1 / rank)
    else:
        reciprocal_ranks.append(0)

    rank_str = str(rank) if rank else "MISS"
    hit_str = "Yes" if hit else "No"
    print(f"  {test['question'][:48]:<50} {test['expected_source']:<35} {rank_str:>5} {hit_str:>5}")

print()
print(f"  Hit Rate:              {hits}/{len(test_queries)} ({hits/len(test_queries)*100:.0f}%)")
mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
print(f"  Mean Reciprocal Rank:  {mrr:.3f}")

# Show what the MISSes actually retrieved
miss_queries = [t for t in test_queries if t["question"] in
    ["What are the WFH rules?", "How much PTO do new hires get?",
     "What initiatives are planned for Q1 2025?"]]

print(f"\n{'='*70}")
print("MISS ANALYSIS — What did the pipeline retrieve instead?")
print(f"{'='*70}")

for mq in miss_queries:
    sources = naive_retrieve(mq["question"], top_k=3)
    # Check if this one actually missed
    found = any(s["metadata"].get("source") == mq["expected_source"] for s in sources)
    if found:
        continue  # Only show actual misses at top_k=5

    sources_5 = naive_retrieve(mq["question"], top_k=5)
    found_5 = any(s["metadata"].get("source") == mq["expected_source"] for s in sources_5)
    if found_5:
        continue

    print(f"\n  Query:    {mq['question']}")
    print(f"  Expected: {mq['expected_source']}")
    print(f"  Got instead:")
    for i, src in enumerate(sources):
        score = src["score"]
        source_name = src["metadata"].get("source", "?")
        text_preview = src["text"][:80].replace("\n", " ")
        print(f"    [{i+1}] {score:.4f}  {source_name}")
        print(f"           {text_preview}...")

**What the scores tell you:**

- **Hit Rate** measures whether the right document shows up at all in the top-k. If it's not there, no amount of prompt engineering can produce a correct answer. Wrong chunks in = wrong answer out.
- **MRR** measures how high the right document ranks. A right document at position 5 of 5 is less useful than one at position 1 (the "lost in the middle" effect).
- These metrics only check **retrieval**. They do not check whether Claude's answer is correct. That is a harder problem — one we address in Lab 2.

**Why did "WFH" and "PTO" miss?**

Look at the MISS analysis above. "WFH" is an abbreviation — the remote work policy never uses that term. "PTO" maps to the HR committee discussion *about* the policy, not the policy itself. The embedding model matched vocabulary, not intent.

This is the same vocabulary mismatch problem you saw in Session 2.1's embedding comparison. The difference: in 2.1 it was a measurement curiosity. Here it's a pipeline failure that produces wrong answers.

**What would fix this?** Two approaches (both coming in Sessions 3.2 and beyond):
1. **Metadata filtering** — restrict retrieval to specific document types before searching
2. **Query enrichment** — expand "WFH" to "work from home, remote work" before embedding

When we add metadata filtering in Session 3.2, we will run the **same test queries** and compare Hit Rate and MRR. That is how you prove an improvement with data, not vibes.

---

## 10. Bridge to Filtered RAG

Naive RAG treats all documents equally. Every query gets the same retrieval logic: embed the question, find the closest chunks, done.

But think about the failures we saw:

| Failure | Root Cause | What Would Fix It |
|---------|-----------|-------------------|
| Wrong version (2023 vs 2025 policy) | No temporal filtering | Filter by year or effective date |
| PTO carryover confusion | Conflicting documents treated equally | Filter by document type (memo vs policy) or recency |
| Over-retrieval on vague queries | No category scoping | Filter by doc_type before searching |
| Missing context across documents | Single-pass retrieval | Targeted retrieval from specific document types |
| WFH / PTO vocabulary mismatch | Abbreviations not in corpus | Query enrichment or synonym expansion |

**What if we could tell the retriever what to look for?**

Instead of:
> "Find the 5 most similar chunks."

We could say:
> "Find the 5 most similar chunks **from HR policy documents written after 2024**."

That is metadata-aware retrieval. It uses the metadata you attached to chunks in Session 2.2 -- source, doc_type, date -- to **filter before searching**.

**Session 3.2 builds that pipeline. Lab 2 proves it works using data.**

The failures you documented today are the starting point for Lab 2. You already know where naive RAG breaks. Next session, you build the fix.

---

## 11. Session Recap

### What we built and explored today

1. **First RAG query** -- saw `naive_rag()` in action end-to-end and inspected the full `RAGResponse` with answer, sources, and token counts
2. **Built `build_prompt()`** -- implemented the augmentation stage: formatting retrieved chunks into a grounded prompt with system instructions, source labels, and the user's question
3. **Built `naive_rag()`** -- wired the full pipeline: retrieve → augment → generate, including the empty-results edge case
4. **Answer vs Evidence panel** -- used `compare_answer_to_evidence()` to verify whether Claude's claims are supported by the retrieved chunks
5. **The Confidence Mirage** -- saw the most dangerous failure: high scores, confident answer, wrong result (PTO carryover conflict)
6. **Failure Gallery** -- systematically broke the pipeline across five failure categories
7. **Retrieval Anatomy** -- visualized score landscapes, saw how source concentration and vocabulary mismatch affect what Claude sees
8. **Top-K Deep Dive** -- proved that k is a correctness knob, not just a cost knob
9. **Scored retrieval quality** -- measured Hit Rate and MRR on a mixed test set, saw real misses from vocabulary mismatch

### Key takeaways

- **Retrieval quality determines answer quality.** Wrong chunks in = wrong answer out.
- **Prompt design determines grounding.** Without explicit "say I don't know" instructions, Claude fills gaps from training data.
- **The evidence chain matters.** Question, chunks, scores, answer -- you need all of it to diagnose problems.
- **The dangerous failures look correct.** Low scores are easy to catch. High scores with wrong answers are the real threat.
- **Naive RAG is your floor.** It works for straightforward factual questions. It breaks on temporal, cross-document, vocabulary mismatch, and ambiguous queries.
- **Vocabulary mismatch is the #1 retrieval failure.** Same concept, different words = MISS. This is the embedding fails lesson from 2.1 showing up in production.

### Questions to leave with

Think about these before Session 3.2:

- **On Filtering:** Every query gets the same retrieval logic -- what if the question type matters? Could we filter before we search?
- **On Comparison:** How would you prove that metadata filtering actually improves results? What does "better" even mean?
- **On Measurement:** If you had 100 questions, how would you score your RAG pipeline without reading every answer yourself?

### What is next

| Session | What We Build | Status |
|---------|--------------|--------|
| **1.1** | API connection + generation | Done |
| **1.2** | Structured extraction with Pydantic | Done |
| **2.1** | Embeddings + similarity measurement | Done |
| **2.2** | Chunking + vector store ingestion | Done |
| **3.1** | Naive RAG -- build_prompt + naive_rag | **Done (today)** |
| **3.2** | Metadata-filtered RAG + evaluation | Next session |
| **4.1** | Observability and debugging | Coming |
| **4.2** | Module test | Coming |

### Homework: Naive RAG Stress Test (not graded -- preparation for Lab 2)

Before next session:

1. Write **10 test questions** (3 should-work, 3 should-struggle, 2 ambiguous, 2 unanswerable)
2. Run all 10 through `naive_rag()` and record results
3. Rate each: **Good** (correct and grounded), **Partial** (partially correct), or **Bad** (wrong/hallucinated)
4. Experiment with `top_k=1, 3, 5, 10` on 2 questions

Save as `experiments/session_3_1_rag_stress_test.md`

**Lab 1 reminder:** Due at end of Session 3.2. If you have not started, start tonight.